# VisDrone Tiny Fixture Smoke Test

Очень быстрый самодостаточный тест основных модулей проекта. Ноутбук создаёт мини-датасет в YOLO-структуре с классами VisDrone и маленькими bbox, затем проверяет validation, analyzer, rule engine, object bank, custom augmentations, COCO conversion/eval и AutoAug-like candidate generator.

Ожидаемое время: обычно меньше 1 минуты на CPU.

In [ ]:
from pathlib import Path
import json, sys, time

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / 'src').exists() else cwd.parent
assert (PROJECT_ROOT / 'src').exists(), f'Cannot find project root from {cwd}'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
from src.analysis.dataset_analyzer import DatasetAnalyzerConfig, analyze_dataset
from src.augmentation.albumentations_transforms import apply_custom_transforms
from src.augmentation.object_bank import ObjectBank
from src.augmentation.policy_to_ultralytics import AugmentationPolicy
from src.data.visdrone_fixture import VISDRONE_CLASS_NAMES, create_predictions_from_gt, create_visdrone_tiny_fixture
from src.data.visdrone_manager import validate_visdrone_yolo_structure
from src.evaluation.coco_converter import convert_yolo_gt_to_coco, convert_yolo_pred_txt_to_coco
from src.evaluation.coco_eval_runner import run_coco_eval
from src.evaluation.metrics_report import build_markdown_report
from src.experiments.autoaug_search import generate_autoaug_candidates, save_autoaug_candidates
from src.policy.rule_engine import RuleEngineConfig, generate_policy_from_stats, save_policy_artifacts
from src.utils.io import dump_json

ROOT = PROJECT_ROOT / 'artifacts' / 'visdrone_tiny_fixture_smoke'
DATASET_ROOT = ROOT / 'dataset'
STATS_DIR = ROOT / 'stats'
POLICY_DIR = ROOT / 'policy'
EVAL_DIR = ROOT / 'eval'
REPORT_DIR = PROJECT_ROOT / 'artifacts' / 'reports'
for path in [STATS_DIR, POLICY_DIR, EVAL_DIR, REPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

timings = {}
t0 = time.perf_counter()

In [ ]:
start = time.perf_counter()
create_visdrone_tiny_fixture(DATASET_ROOT, train_images=12, val_images=4)
timings['fixture_build'] = time.perf_counter() - start

validation = validate_visdrone_yolo_structure(DATASET_ROOT, splits=('train', 'val'))
assert validation['is_valid'], validation
print(json.dumps(validation['splits'], indent=2, ensure_ascii=False))

In [ ]:
start = time.perf_counter()
stats = analyze_dataset(
    dataset_root=DATASET_ROOT,
    output_dir=STATS_DIR,
    splits=('train', 'val'),
    config=DatasetAnalyzerConfig(generate_plots=False, show_progress=False),
)
timings['analysis'] = time.perf_counter() - start
assert stats['splits']['train']['num_objects'] > 0
assert stats['splits']['train']['ratios']['small_ratio'] > 0.5
print('small_ratio =', stats['splits']['train']['ratios']['small_ratio'])

In [ ]:
start = time.perf_counter()
policy, decision_report = generate_policy_from_stats(stats, cfg=RuleEngineConfig())
paths = save_policy_artifacts(policy, decision_report, output_dir=POLICY_DIR)
timings['policy_generation'] = time.perf_counter() - start
assert policy['albumentations_spec']
assert decision_report['fired_rules']
print('policy paths =', {k: str(v) for k, v in paths.items()})
print('fired rules =', [x['rule_name'] for x in decision_report['fired_rules']])

In [ ]:
start = time.perf_counter()
bank = ObjectBank(seed=42)
bank.build_from_dataset(DATASET_ROOT / 'images' / 'train', DATASET_ROOT / 'labels' / 'train')
bank_path = POLICY_DIR / 'object_bank.json'
bank.save(bank_path)
loaded_bank = ObjectBank.load(bank_path, seed=42)
timings['object_bank'] = time.perf_counter() - start
assert loaded_bank.size > 0
print('object bank size =', loaded_bank.size)

In [ ]:
import cv2
from src.data.yolo_label_reader import load_yolo_labels, yolo_bbox_to_xyxy_px

image_path = sorted((DATASET_ROOT / 'images' / 'train').glob('*.jpg'))[0]
image = cv2.imread(str(image_path))
h, w = image.shape[:2]
labels = load_yolo_labels(DATASET_ROOT / 'labels' / 'train' / f'{image_path.stem}.txt')
sample = {
    'image': image,
    'bboxes': [list(yolo_bbox_to_xyxy_px(b, image_width=w, image_height=h)) for b in labels],
    'class_labels': [b.class_id for b in labels],
}
aug_policy = AugmentationPolicy(policy)
transforms = aug_policy.get_albumentations_transforms(object_bank=loaded_bank, seed=42)
out = apply_custom_transforms(sample, transforms)
assert len(out['bboxes']) == len(out['class_labels'])
assert all(0 <= box[0] < box[2] <= out['image'].shape[1] for box in out['bboxes'])
assert all(0 <= box[1] < box[3] <= out['image'].shape[0] for box in out['bboxes'])
print('custom transforms ok; boxes:', len(sample['bboxes']), '->', len(out['bboxes']))

In [ ]:
start = time.perf_counter()
pred_dir = create_predictions_from_gt(DATASET_ROOT / 'labels' / 'val', EVAL_DIR / 'pred_labels')
coco_gt = convert_yolo_gt_to_coco(DATASET_ROOT / 'images' / 'val', DATASET_ROOT / 'labels' / 'val', VISDRONE_CLASS_NAMES, EVAL_DIR / 'coco_gt.json')
coco_dt = convert_yolo_pred_txt_to_coco(pred_dir, DATASET_ROOT / 'images' / 'val', EVAL_DIR / 'coco_dt.json')
metrics = run_coco_eval(EVAL_DIR / 'coco_gt.json', EVAL_DIR / 'coco_dt.json', EVAL_DIR / 'coco_eval.json', use_tiny_eval=True)
timings['coco_eval'] = time.perf_counter() - start
assert len(coco_gt['images']) == 4
assert len(coco_dt) > 0
assert 'AP_small' in metrics
print(json.dumps(metrics, indent=2, ensure_ascii=False))

In [ ]:
candidates = generate_autoaug_candidates(num_candidates=3, seed=42)
candidate_paths = save_autoaug_candidates(candidates, ROOT / 'autoaug_candidates')
assert len(candidates) == 3
print(candidate_paths)

In [ ]:
timings['total'] = time.perf_counter() - t0
report = build_markdown_report(
    {'visdrone_tiny_fixture_gt_predictions': metrics},
    output_path=REPORT_DIR / 'visdrone_tiny_fixture_smoke_report.md',
    timings=timings,
    artifact_paths={
        'dataset_stats': str(STATS_DIR / 'dataset_stats.json'),
        'policy': str(POLICY_DIR / 'policy_adaptive.json'),
        'decision_report': str(POLICY_DIR / 'decision_report.json'),
        'object_bank': str(bank_path),
        'coco_eval': str(EVAL_DIR / 'coco_eval.json'),
        'autoaug_manifest': candidate_paths['manifest'],
    },
)
dump_json({'timings': timings, 'metrics': metrics}, REPORT_DIR / 'visdrone_tiny_fixture_smoke_summary.json')
print('SMOKE TEST OK')
print('report =', report)
print(json.dumps(timings, indent=2))